# Version 2.0 Hillslope-Control Dataset Extraction
## Himalayan Landslide Intelligence — Phase 3b GEE Pipeline

**Purpose:** Generate genuine hillslope control samples (unfailed terrain with $6.2^\circ \le \text{slope} \le 44.5^\circ$)
to replace historical flat-terrain controls (slope $< 5^\circ$) and eliminate the terrain confound.

**Earth Engine Project:** `landslidedetection-506008`
**Target ROI:** $[26.0^\circ\text{N}, 35.0^\circ\text{N}]$, $[74.0^\circ\text{E}, 97.0^\circ\text{E}]$
**Positive Events:** 127 confirmed events from `outputs/phase2_cleaned_dataset.csv`

**Positional Uncertainty Remediation:**
Because NASA Global Landslide Catalog coordinates have up to $\pm 5\text{ km}$ positional uncertainty,
this notebook extracts BOTH point-level values and **neighbourhood buffer reductions** (500m & 1000m buffers)
to capture localized terrain ruggedness and radar backscatter disturbance.

In [ ]:
# 1. Setup & Earth Engine Authentication
!pip install -q geemap pyproj
import ee
import geemap.core as geemap
import pandas as pd
import numpy as np
import time

ee.Authenticate()
ee.Initialize(project='landslidedetection-506008')
print('GEE Initialized successfully:', ee.Number(1).add(1).getInfo() == 2)

In [ ]:
# 2. Load Positive Events and Define ROI
# Upload phase2_cleaned_dataset.csv to Colab working directory
df_p2 = pd.read_csv('phase2_cleaned_dataset.csv')
positives = df_p2[df_p2['label'] == 1].reset_index(drop=True)
print(f'Loaded {len(positives)} positive events.')

lat_min, lat_max = 26.0, 35.0
lon_min, lon_max = 74.0, 97.0
roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])

# SRTM & Slope
srtm = ee.Image('USGS/SRTMGL1_003').select('elevation').clip(roi)
slope = ee.Terrain.slope(srtm)
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')

# Version 2.0 Hillslope Mask: 6.2 deg <= slope <= 44.5 deg, Water occurrence < 10%
hillslope_mask = (
    slope.gte(6.2)
    .And(slope.lte(44.5))
    .And(jrc.unmask(0).lt(10))
)

In [ ]:
# 3. Sample Version 2.0 Hillslope Controls
# Buffer 10-40km around each positive, excluding 2000m around ALL positives
all_pos_pts = [ee.Geometry.Point([r.longitude, r.latitude]) for _, r in positives.iterrows()]
pos_exclusion_zone = ee.FeatureCollection([ee.Feature(p.buffer(2000)) for p in all_pos_pts]).geometry()

v2_controls = []
for idx, row in positives.iterrows():
    p_pt = ee.Geometry.Point([row['longitude'], row['latitude']])
    # Sample candidate from 10km to 40km ring within the same physiographic sub-catchment
    search_ring = p_pt.buffer(40000).difference(p_pt.buffer(10000))
    search_valid = search_ring.difference(pos_exclusion_zone)
    
    # Sample from hillslope mask
    sampled = hillslope_mask.selfMask().sample(
        region=search_valid, scale=90, numPixels=1, seed=42 + idx, geometries=True
    )
    feats = sampled.getInfo()['features']
    if feats:
        coords = feats[0]['geometry']['coordinates']
        v2_controls.append({
            'latitude': coords[1],
            'longitude': coords[0],
            'event_date': row['event_date'],
            'Location Accuracy': 'Unfailed Hillslope Control Point',
            'Landslide Trigger': 'Hillslope Control Match',
            'Imported Event Source Catalog': 'V2_GEE_Hillslope',
            'label': 0,
            'block_id': f"{int(coords[0]//0.5)}_{int(coords[1]//0.5)}",
            'paired_positive_idx': idx
        })
    else:
        print(f'Warning: No hillslope control found for point {idx}')

df_v2_controls = pd.DataFrame(v2_controls)
print(f'Successfully sampled {len(df_v2_controls)} Version 2.0 hillslope controls.')
df_v2_controls.to_csv('phase2_v2_hillslope_controls.csv', index=False)

# 4. Multi-Scale Feature Extraction Loop for V2 Controls
Extract static topographic and dynamic features with both point-sample and 500m circular buffer reductions:
```python
# Buffer reduction example to address +/- 5km positional uncertainty:
# buffer_500m = point.buffer(500)
# sar_mean = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=buffer_500m, scale=10)
# sar_min = img.reduceRegion(reducer=ee.Reducer.min(), geometry=buffer_500m, scale=10)
```
Save output to `artifacts/susceptibility/v2_feature_table_experimental.csv`.